# FastFHIR vs JSON-FHIR Benchmark Results

This notebook loads and analyzes benchmark results from the C++ harness.

In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import os
from sqlalchemy import create_engine, text

# Configuration - read from environment or use defaults
db_host = os.getenv('POSTGRES_HOST', 'localhost')
db_port = os.getenv('POSTGRES_PORT', '5432')
db_name = os.getenv('POSTGRES_DB', 'fhir_benchmark')
db_user = os.getenv('POSTGRES_USER', 'postgres')
db_pass = os.getenv('POSTGRES_PASSWORD', 'postgres')

db_url = f"postgresql+psycopg2://{db_user}:{db_pass}@{db_host}:{db_port}/{db_name}"
print(f"Connecting to {db_user}@{db_host}:{db_port}/{db_name}")

Connecting to bench@localhost:5432/benchmark


## Load Results from Database

In [16]:
try:
    engine = create_engine(db_url, pool_pre_ping=True)
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("Connected successfully (SQLAlchemy)")
except Exception as e:
    print(f"Connection failed: {e}")
    engine = None

Connected successfully (SQLAlchemy)


## Summary Statistics

In [17]:
if 'engine' in locals() and engine is not None:
    query = """
    SELECT 
        br.id,
        run_id,
        arm,
        stage,
        duration_us,
        target_mb,
        patients_in_bundle,
        br.created_at
    FROM benchmark_results br
    ORDER BY run_id DESC, target_mb, arm, stage
    LIMIT 10000
    """
    
    # Store full result set for analysis (no automatic printing).
    df = pd.read_sql(query, engine)
    
    # Convenience DataFrames for manual inspection when needed.
    df_preview = df.head(20).copy()
    latest_run_id = int(df['run_id'].max()) if len(df) > 0 else None
    latest_run_df = df[df['run_id'] == latest_run_id].copy() if latest_run_id is not None else pd.DataFrame()
else:
    df = pd.DataFrame()
    df_preview = pd.DataFrame()
    latest_run_df = pd.DataFrame()
    latest_run_id = None

## Summary Statistics

In [18]:
if 'df' in locals() and len(df) > 0:
    summary = df.groupby(['arm', 'stage'])['duration_us'].agg(['mean', 'std', 'min', 'max']).round(2)
    arm_totals = df.groupby('arm')['duration_us'].agg(['sum', 'mean', 'count']).round(2)
    
    # Keep optional preview frames ready for manual review.
    summary_preview = summary.reset_index().copy()
    arm_totals_preview = arm_totals.reset_index().copy()
else:
    summary = pd.DataFrame()
    arm_totals = pd.DataFrame()
    summary_preview = pd.DataFrame()
    arm_totals_preview = pd.DataFrame()

## Performance Comparison - Serialization

In [ ]:
if 'df' in locals() and len(df) > 0:
    stage1_data = df[df['stage'] == 'stage1_serialize'].copy()
    
    if len(stage1_data) > 0:
        fig, ax = plt.subplots(figsize=(12, 6))
        for arm in stage1_data['arm'].unique():
            data = stage1_data[stage1_data['arm'] == arm].sort_values('target_mb')
            ax.plot(data['target_mb'], data['duration_us'], marker='o', label=arm, linewidth=2)
        
        ax.set_xlabel('Bundle Size (MB)', fontsize=12)
        ax.set_ylabel('Serialization Time (microseconds)', fontsize=12)
        ax.set_title('Serialization Performance by Arm', fontsize=14, fontweight='bold')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.set_xscale('log')
        ax.set_yscale('log')
        plt.tight_layout()
        plt.show()
    else:
        print("No serialization data found")

## Performance Comparison - Query

In [ ]:
if 'df' in locals() and len(df) > 0:
    stage3_data = df[df['stage'] == 'stage3_query'].copy()
    
    if len(stage3_data) > 0:
        fig, ax = plt.subplots(figsize=(12, 6))
        for arm in stage3_data['arm'].unique():
            data = stage3_data[stage3_data['arm'] == arm].sort_values('target_mb')
            ax.plot(data['target_mb'], data['duration_us'], marker='s', label=arm, linewidth=2)
        
        ax.set_xlabel('Bundle Size (MB)', fontsize=12)
        ax.set_ylabel('Query Time (microseconds)', fontsize=12)
        ax.set_title('Query Performance by Arm', fontsize=14, fontweight='bold')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.set_xscale('log')
        ax.set_yscale('log')
        plt.tight_layout()
        plt.show()
    else:
        print("No query data found")

## Cleanup

In [ ]:
if 'engine' in locals() and engine is not None:
    engine.dispose()
    print("Database engine disposed")